In [1]:
%pip install pandas numpy 

  Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.4-cp314-cp314-macosx_14_0_arm64.whl (5.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import re

INPUT_PATH = "../data/processed/extracted_data.csv"
OUTPUT_PATH = "../data/processed/cleaned_data.csv"

# ------------------------------------------
# Load Dataset
# ------------------------------------------

df = pd.read_csv(INPUT_PATH)

print("Initial Shape:", df.shape)

print("\nInitial Missing Values:")
print(df.isnull().sum())

# ==========================================
# STANDARDIZE COLUMN NAMES
# ==========================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# ==========================================
# REMOVE DUPLICATES
# ==========================================

df.drop_duplicates(inplace=True)

# ==========================================
# CLEAN NAME COLUMN
# ==========================================

def clean_name(name):
    if pd.isna(name):
        return np.nan

    name = str(name).strip().title()
    name = re.sub(r"\s+", " ", name)

    # convert "SMITH, ALICE" -> "Alice Smith"
    if "," in name:
        parts = [x.strip() for x in name.split(",")]
        if len(parts) == 2:
            name = f"{parts[1]} {parts[0]}"

    return name

df["name"] = df["name"].apply(clean_name)

# ==========================================
# CLEAN AGE
# ==========================================

df["age"] = pd.to_numeric(df["age"], errors="coerce")

# invalid ages
df.loc[(df["age"] < 0) | (df["age"] > 120), "age"] = np.nan

# fill missing ages with median
df["age"] = df["age"].fillna(df["age"].median())

# ==========================================
# CLEAN GENDER
# ==========================================

gender_map = {
    "m": "male",
    "male": "male",
    "f": "female",
    "female": "female",
    "unknown": np.nan
}

df["gender"] = (
    df["gender"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(gender_map)
)

df["gender"] = df["gender"].fillna("unknown")

# ==========================================
# CLEAN CITY
# ==========================================

city_map = {
    "newyork": "new york",
    "nwe yrok": "new york",
    "la": "los angeles"
}

df["city"] = (
    df["city"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["city"] = df["city"].replace(city_map)

# fill missing city
df["city"] = df["city"].replace("nan", np.nan)
df["city"] = df["city"].fillna("unknown")

# ==========================================
# CLEAN BMI
# ==========================================

def clean_bmi(value):

    if pd.isna(value):
        return np.nan

    value = str(value).lower()

    # extract numeric part
    match = re.search(r"(\d+\.?\d*)", value)

    if match:
        return float(match.group(1))

    return np.nan

df["bmi"] = df["bmi"].apply(clean_bmi)

# invalid bmi
df.loc[(df["bmi"] < 10) | (df["bmi"] > 60), "bmi"] = np.nan

df["bmi"] = df["bmi"].fillna(df["bmi"].median())

# ==========================================
# CLEAN BLOOD PRESSURE
# ==========================================

def extract_bp(bp):

    if pd.isna(bp):
        return np.nan, np.nan

    bp = str(bp).lower()

    numbers = re.findall(r"\d+", bp)

    if len(numbers) >= 2:
        return int(numbers[0]), int(numbers[1])

    return np.nan, np.nan

df[["systolic_bp", "diastolic_bp"]] = (
    df["blood_pressure"]
    .apply(lambda x: pd.Series(extract_bp(x)))
)

# fill missing BP values
df["systolic_bp"] = df["systolic_bp"].fillna(df["systolic_bp"].median())
df["diastolic_bp"] = df["diastolic_bp"].fillna(df["diastolic_bp"].median())

# ==========================================
# CLEAN HEART RATE
# ==========================================

def clean_heart_rate(hr):

    if pd.isna(hr):
        return np.nan

    hr = str(hr).lower().strip()

    if hr == "eighty":
        return 80

    try:
        hr = int(hr)

        if 30 <= hr <= 220:
            return hr

    except:
        return np.nan

    return np.nan

df["heart_rate"] = df["heart_rate"].apply(clean_heart_rate)

df["heart_rate"] = df["heart_rate"].fillna(df["heart_rate"].median())

# ==========================================
# CLEAN CHOLESTEROL
# ==========================================

cholesterol_map = {
    "normal": 180,
    "high": 240
}

def clean_cholesterol(value):

    if pd.isna(value):
        return np.nan

    value = str(value).lower().strip()

    if value in cholesterol_map:
        return cholesterol_map[value]

    try:
        return float(value)

    except:
        return np.nan

df["cholesterol_level"] = (
    df["cholesterol_level"]
    .apply(clean_cholesterol)
)

df["cholesterol_level"] = (
    df["cholesterol_level"]
    .fillna(df["cholesterol_level"].median())
)

# ==========================================
# CLEAN DIABETIC COLUMN
# ==========================================

diabetic_map = {
    "yes": 1,
    "y": 1,
    "no": 0,
    "n": 0,
    "unknown": np.nan
}

df["diabetic"] = (
    df["diabetic"]
    .astype(str)
    .str.lower()
    .str.strip()
    .map(diabetic_map)
)

df["diabetic"] = df["diabetic"].fillna(0)

# ==========================================
# CLEAN SMOKER COLUMN
# ==========================================

def clean_smoker(value):

    if pd.isna(value):
        return "unknown"

    value = str(value).lower().strip()

    if value in ["yes", "smoker"]:
        return "smoker"

    if value in ["no", "non-smoker"]:
        return "non-smoker"

    if value in ["former", "ex-smoker"]:
        return "former smoker"

    return "unknown"

df["smoker"] = df["smoker"].apply(clean_smoker)

# ==========================================
# CLEAN MEDICATIONS
# ==========================================

df["medications"] = (
    df["medications"]
    .fillna("none")
    .astype(str)
    .str.lower()
    .str.replace(";", ",")
)

# ==========================================
# CLEAN DATES
# ==========================================

df["last_visit_date"] = pd.to_datetime(
    df["last_visit_date"],
    errors="coerce"
)

# ==========================================
# CLEAN FOLLOW UP
# ==========================================

df["follow_up"] = pd.to_numeric(
    df["follow_up"],
    errors="coerce"
)

df["follow_up"] = df["follow_up"].fillna(0)

# ==========================================
# CLEAN DIAGNOSIS CODE
# ==========================================

df["diagnosis_code"] = (
    df["diagnosis_code"]
    .fillna("unknown")
    .astype(str)
    .str.upper()
)

# ==========================================
# CLEAN NOTES
# ==========================================

df["notes"] = (
    df["notes"]
    .fillna("no notes")
    .astype(str)
)

# remove emojis
df["notes"] = df["notes"].str.replace(
    r"[^\x00-\x7F]+",
    "",
    regex=True
)

# ==========================================
# CLEAN TARGET COLUMN
# ==========================================

target_map = {
    "1": 1,
    "0": 0,
    "unknown": np.nan
}

df["has_disease"] = (
    df["has_disease"]
    .astype(str)
    .str.lower()
    .map(target_map)
)

df["has_disease"] = df["has_disease"].fillna(0)

# ==========================================
# DROP UNUSED COLUMN
# ==========================================

df.drop(columns=["blood_pressure"], inplace=True)

# ==========================================
# FINAL CHECK
# ==========================================

print("\nFinal Missing Values:")
print(df.isnull().sum())

print("\nFinal Shape:")
print(df.shape)

print("\nCleaned Data Preview:")
print(df.head())

# ==========================================
# SAVE CLEANED DATA
# ==========================================

df.to_csv(OUTPUT_PATH, index=False)

print("\nCleaned dataset saved successfully.")

Initial Shape: (10000, 17)

Initial Missing Values:
Patient_ID            507
Name                    0
Age                     0
Gender               1707
City                 1003
BMI                  2418
Blood_Pressure       2030
Heart_Rate              0
Cholesterol_Level    2052
Diabetic             1641
Smoker               3234
Medications          4024
Last_Visit_Date         0
Follow_Up            2524
Diagnosis_Code       1675
Notes                4024
Has_Disease          2520
dtype: int64

Final Missing Values:
patient_id            507
name                    0
age                     0
gender                  0
city                    0
bmi                     0
heart_rate              0
cholesterol_level       0
diabetic                0
smoker                  0
medications             0
last_visit_date      7508
follow_up               0
diagnosis_code          0
notes                   0
has_disease             0
systolic_bp             0
diastolic_bp            0
dt